
# Vanilla Zero-shot Baseline — ASAP Essay Sets 5–8

Notebook:
- đọc `dataset/asap/asap_train.csv`, `dataset/asap/asap_val.csv`, `dataset/asap/asap_test.csv`
- concat 3 file
- shuffle và lấy 10% evaluation
- chỉ chạy Prompt 5–8
- gọi vanilla Mistral 7B chấm điểm
- tính QWK theo từng prompt và macro average


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

NEED_RESTART_FLAG = "/content/.deps_installed_for_vanilla_mistral"

if not os.path.exists(NEED_RESTART_FLAG):
    print("Installing safe dependency set...")

    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", "--no-cache-dir", "--upgrade", "--no-deps",
        "sympy==1.14.0",
        "huggingface_hub==0.24.7",
        "tokenizers==0.19.1",
        "safetensors==0.4.5",
        "transformers==4.44.2",
        "accelerate==0.33.0",
        "bitsandbytes==0.43.3",
    ])

    Path(NEED_RESTART_FLAG).write_text("installed", encoding="utf-8")

    print("Install done. Restarting runtime now...")
    os.kill(os.getpid(), 9)

print("Dependencies already installed. Continue imports...")

import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

from google.colab import drive
drive.mount("/content/drive")

print("Setup OK")
print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

Dependencies already installed. Continue imports...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup OK
torch: 2.10.0+cu128
numpy: 2.0.2
pandas: 2.2.2


In [2]:
from pathlib import Path

# Root repo trong Google Drive
REPO_DIR = Path("/content/drive/MyDrive/ielts/IELTS-Writing-Evals")

# Dataset path theo ảnh bạn gửi
DATA_DIR = REPO_DIR / "dataset" / "asap"

TRAIN_PATH = DATA_DIR / "asap_train.csv"
VAL_PATH   = DATA_DIR / "asap_val.csv"
TEST_PATH  = DATA_DIR / "asap_test.csv"

ESSAY_SETS_TO_RUN = [5, 6, 7, 8]

EVAL_FRAC = 0.10
RANDOM_SEED = 222

# Test nhanh thì đặt ví dụ 30. Chạy thật thì để None.
MAX_EVAL_PER_SET = None

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

MAX_NEW_TOKENS = 32
TEMPERATURE = 0.0
DO_SAMPLE = False

# Output lưu thẳng vào repo để dễ tìm
OUTPUT_DIR = REPO_DIR / "vanilla_baseline_outputs_p5_p8"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", Path.cwd())
print("REPO_DIR:  ", REPO_DIR, REPO_DIR.exists())
print("DATA_DIR:  ", DATA_DIR, DATA_DIR.exists())
print("TRAIN_PATH:", TRAIN_PATH, TRAIN_PATH.exists())
print("VAL_PATH:  ", VAL_PATH, VAL_PATH.exists())
print("TEST_PATH: ", TEST_PATH, TEST_PATH.exists())

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Không tìm thấy {p}. Kiểm tra lại tên folder Drive: "
            "MyDrive/ielts/IELTS-Writing-Evals/dataset/asap"
        )

Current working directory: /content
REPO_DIR:   /content/drive/MyDrive/ielts/IELTS-Writing-Evals True
DATA_DIR:   /content/drive/MyDrive/ielts/IELTS-Writing-Evals/dataset/asap True
TRAIN_PATH: /content/drive/MyDrive/ielts/IELTS-Writing-Evals/dataset/asap/asap_train.csv True
VAL_PATH:   /content/drive/MyDrive/ielts/IELTS-Writing-Evals/dataset/asap/asap_val.csv True
TEST_PATH:  /content/drive/MyDrive/ielts/IELTS-Writing-Evals/dataset/asap/asap_test.csv True



## 1. Load và concat 3 split


In [3]:

def read_table(path):
    path = Path(path)
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    return pd.read_csv(path)

train_df = read_table(TRAIN_PATH)
val_df = read_table(VAL_PATH)
test_df = read_table(TEST_PATH)

train_df["source_split"] = "train"
val_df["source_split"] = "val"
test_df["source_split"] = "test"

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Shape train:", train_df.shape)
print("Shape val:  ", val_df.shape)
print("Shape test: ", test_df.shape)
print("Shape all:  ", all_df.shape)
print("\nColumns:")
print(all_df.columns.tolist())

all_df.head()


Shape train: (9084, 6)
Shape val:   (1296, 6)
Shape test:  (2596, 6)
Shape all:   (12976, 6)

Columns:
['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm', 'source_split']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm,source_split
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75,train
1,9985,4,The author concluded this sentence because he ...,0.0,0.00,train
2,3231,2,"I can think of several books that, I would not...",4.0,0.60,train
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65,train
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77,train



## 2. Tự nhận dạng cột essay set, essay text, gold score


In [4]:

def find_col(df, candidates, required=True):
    normalized = {c.lower().strip().replace(" ", "_"): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip().replace(" ", "_")
        if key in normalized:
            return normalized[key]
    for cand in candidates:
        key = cand.lower().strip().replace(" ", "_")
        for norm, original in normalized.items():
            if key in norm:
                return original
    if required:
        raise KeyError(f"Không tìm thấy cột trong candidates={candidates}. Columns={df.columns.tolist()}")
    return None

ESSAY_SET_COL = find_col(all_df, ["essay_set", "prompt_id", "set", "essayset"])
ESSAY_COL = find_col(all_df, ["essay", "full_text", "text", "response", "essay_text"])

def pick_gold_column(df, essay_set_id):
    # P5/P6: Final score 0-4
    # P7: Resolved_Score 0-30
    # P8: Resolved Score 0-60
    if essay_set_id in [5, 6]:
        candidates = [
            "final", "Final", "domain1_score", "resolved_score", "Resolved Score",
            "score", "final_score"
        ]
    elif essay_set_id == 7:
        candidates = [
            "resolved_score", "Resolved_Score", "Resolved Score",
            "domain1_score", "score", "final_score"
        ]
    elif essay_set_id == 8:
        candidates = [
            "resolved_score", "Resolved Score", "Resolved_Score",
            "domain1_score", "score", "final_score"
        ]
    else:
        candidates = ["domain1_score", "resolved_score", "score", "final_score"]
    return find_col(df, candidates, required=True)

print("ESSAY_SET_COL:", ESSAY_SET_COL)
print("ESSAY_COL:    ", ESSAY_COL)

for s in ESSAY_SETS_TO_RUN:
    sub = all_df[all_df[ESSAY_SET_COL].astype(int) == s]
    print(f"P{s}: rows={len(sub)}, gold_col={pick_gold_column(sub, s) if len(sub) else 'N/A'}")


ESSAY_SET_COL: essay_set
ESSAY_COL:     essay
P5: rows=1805, gold_col=domain1_score
P6: rows=1800, gold_col=domain1_score
P7: rows=1569, gold_col=domain1_score
P8: rows=723, gold_col=domain1_score



## 3. Prompt text và score range cho P5–P8

- P5: 0–4
- P6: 0–4
- P7: 0–30
- P8: 0–60


In [5]:

PROMPT_INFO = {
    5: {
        "score_min": 0,
        "score_max": 4,
        "prompt": """Read the memoir "Narciso Rodriguez" from Home: The Blueprints of Our Lives.

Describe the mood created by the author in the memoir. Support your answer with relevant and specific information from the memoir.""",
        "rubric_short": "Score 0-4. Higher scores give a clearer, more complete, and more accurate description of the mood, with relevant and specific support from the memoir."
    },
    6: {
        "score_min": 0,
        "score_max": 4,
        "prompt": """Read the excerpt "The Mooring Mast" by Marcia Amidon Lusted.

Based on the excerpt, describe the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. Support your answer with relevant and specific information from the excerpt.""",
        "rubric_short": "Score 0-4. Higher scores give a clearer, more complete, and more accurate description of the obstacles, with relevant and specific support from the excerpt."
    },
    7: {
        "score_min": 0,
        "score_max": 30,
        "prompt": """Write about patience. Being patient means that you are understanding and tolerant. A patient person experiences difficulties without complaining.

Do only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.""",
        "rubric_short": "Composite score 0-30. The rubric rates four traits: Ideas with doubled weight, Organization, Style, and Conventions. Higher scores mean focused and developed ideas, logical organization, effective language, varied sentences, and strong conventions."
    },
    8: {
        "score_min": 0,
        "score_max": 60,
        "prompt": """We all understand the benefits of laughter. For example, someone once said, "Laughter is the shortest distance between two people." Many other people believe that laughter is an important part of any relationship.

Tell a true story in which laughter was one element or part.""",
        "rubric_short": "Composite score 0-60. The rubric rates six writing traits, with the final composite emphasizing Ideas and Content, Organization, Sentence Fluency, and Conventions. Higher scores mean clearer ideas, stronger organization, effective voice and word choice, fluent sentences, and strong conventions."
    },
}



## 4. Lấy 10% evaluation sau khi concat train + val + test


In [6]:

df_58 = all_df[all_df[ESSAY_SET_COL].astype(int).isin(ESSAY_SETS_TO_RUN)].copy()
df_58[ESSAY_SET_COL] = df_58[ESSAY_SET_COL].astype(int)

eval_parts = []
for s in ESSAY_SETS_TO_RUN:
    sub = df_58[df_58[ESSAY_SET_COL] == s].copy()
    n = max(1, int(round(len(sub) * EVAL_FRAC)))
    part = sub.sample(n=n, random_state=RANDOM_SEED)
    if MAX_EVAL_PER_SET is not None:
        part = part.sample(n=min(MAX_EVAL_PER_SET, len(part)), random_state=RANDOM_SEED)
    eval_parts.append(part)

eval_df = pd.concat(eval_parts, ignore_index=True)
eval_df = eval_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

print("Total P5-P8 rows:", len(df_58))
print("Eval rows:", len(eval_df))
print(eval_df[ESSAY_SET_COL].value_counts().sort_index())

eval_df[[ESSAY_SET_COL, ESSAY_COL, "source_split"]].head()


Total P5-P8 rows: 5897
Eval rows: 589
essay_set
5    180
6    180
7    157
8     72
Name: count, dtype: int64


,essay_set,essay,source_split
0,7,??? One time when I was being very patient was...,train
1,8,"Day after day, people laugh. For many differen...",test
2,5,"The mood of the memoir was happy because, Narc...",test
3,6,"Despite Al Smith's goal of fulfilling the ""dre...",train
4,6,In the story The Mooring Mast by Marcia Amidon...,train



## 5. Load Vanilla Mistral 7B


In [7]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def load_llm(model_name=MODEL_NAME):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    use_cuda = torch.cuda.is_available()
    print("CUDA available:", use_cuda)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    if use_cuda:
        print("Loading model on GPU with float16, no bitsandbytes...")

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
    else:
        print("Loading model on CPU. This will be very slow...")

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            device_map="cpu",
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )

    model.eval()

    gen = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
    )

    return gen, tokenizer

gen, tokenizer = load_llm()

CUDA available: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading model on GPU with float16, no bitsandbytes...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


## 6. Build prompt, parse điểm, chạy inference


In [8]:

def build_scoring_prompt(essay_set_id, essay_text):
    info = PROMPT_INFO[int(essay_set_id)]
    lo, hi = info["score_min"], info["score_max"]

    user_msg = f"""You are an expert essay scorer.

Essay prompt:
{info["prompt"]}

Rubric summary:
{info["rubric_short"]}

Task:
Score the student's essay with an integer from {lo} to {hi}.
Return only the integer score. Do not explain.

Student essay:
{essay_text}
"""
    return f"<s>[INST] {user_msg.strip()} [/INST]"

def parse_score(raw_text, essay_set_id):
    info = PROMPT_INFO[int(essay_set_id)]
    lo, hi = info["score_min"], info["score_max"]

    text = raw_text.split("[/INST]")[-1].strip()
    nums = re.findall(r"-?\d+", text)
    if not nums:
        return None

    val = int(nums[0])
    return max(lo, min(hi, val))

def score_one_essay(essay_set_id, essay_text):
    prompt = build_scoring_prompt(essay_set_id, essay_text)

    gen_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.eos_token_id,
        "return_full_text": True,
    }
    if DO_SAMPLE:
        gen_kwargs["temperature"] = TEMPERATURE

    out = gen(prompt, **gen_kwargs)[0]["generated_text"]
    pred = parse_score(out, essay_set_id)
    return pred, out


In [9]:

pred_rows = []
checkpoint_path = OUTPUT_DIR / "vanilla_mistral_p5_p8_eval_predictions_checkpoint.csv"

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    essay_set_id = int(row[ESSAY_SET_COL])
    essay_text = str(row[ESSAY_COL])

    gold_col = pick_gold_column(eval_df[eval_df[ESSAY_SET_COL] == essay_set_id], essay_set_id)
    gold = row[gold_col]

    try:
        pred, raw = score_one_essay(essay_set_id, essay_text)
    except Exception as e:
        pred, raw = None, f"ERROR: {repr(e)}"

    pred_rows.append({
        "row_id": i,
        "essay_set": essay_set_id,
        "source_split": row.get("source_split", None),
        "gold_col": gold_col,
        "gold": gold,
        "pred": pred,
        "raw_generation": raw,
        "essay": essay_text,
    })

    if (i + 1) % 10 == 0:
        pd.DataFrame(pred_rows).to_csv(checkpoint_path, index=False)

pred_df = pd.DataFrame(pred_rows)
pred_path = OUTPUT_DIR / "vanilla_mistral_p5_p8_eval_predictions.csv"
pred_df.to_csv(pred_path, index=False)

print("Saved:", pred_path)
pred_df.head()


  0%|          | 0/589 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved: /content/drive/MyDrive/ielts/IELTS-Writing-Evals/vanilla_baseline_outputs_p5_p8/vanilla_mistral_p5_p8_eval_predictions.csv


,row_id,essay_set,source_split,gold_col,gold,pred,raw_generation,essay
0,0,7,train,domain1_score,20.0,22,<s>[INST] You are an expert essay scorer.\n\nE...,??? One time when I was being very patient was...
1,1,8,test,domain1_score,38.0,48,<s>[INST] You are an expert essay scorer.\n\nE...,"Day after day, people laugh. For many differen..."
2,2,5,test,domain1_score,1.0,3,<s>[INST] You are an expert essay scorer.\n\nE...,"The mood of the memoir was happy because, Narc..."
3,3,6,train,domain1_score,4.0,4,<s>[INST] You are an expert essay scorer.\n\nE...,"Despite Al Smith's goal of fulfilling the ""dre..."
4,4,6,train,domain1_score,3.0,4,<s>[INST] You are an expert essay scorer.\n\nE...,In the story The Mooring Mast by Marcia Amidon...



## 7. Tính QWK


In [10]:

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

metric_rows = []
for s in ESSAY_SETS_TO_RUN:
    sub = pred_df[(pred_df["essay_set"] == s) & pred_df["pred"].notna() & pred_df["gold"].notna()].copy()
    sub["gold"] = sub["gold"].astype(int)
    sub["pred"] = sub["pred"].astype(int)

    score = np.nan if len(sub) == 0 else qwk(sub["gold"], sub["pred"])

    metric_rows.append({
        "method": "Vanilla Mistral 7B zero-shot",
        "essay_set": f"P{s}",
        "n": len(sub),
        "qwk": score,
        "gold_min": sub["gold"].min() if len(sub) else np.nan,
        "gold_max": sub["gold"].max() if len(sub) else np.nan,
        "pred_min": sub["pred"].min() if len(sub) else np.nan,
        "pred_max": sub["pred"].max() if len(sub) else np.nan,
    })

metrics_df = pd.DataFrame(metric_rows)
avg_qwk = metrics_df["qwk"].mean(skipna=True)

display(metrics_df)
print("Macro Avg QWK P5-P8:", avg_qwk)

metrics_path = OUTPUT_DIR / "vanilla_mistral_p5_p8_qwk_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)


,method,essay_set,n,qwk,gold_min,gold_max,pred_min,pred_max
0,Vanilla Mistral 7B zero-shot,P5,180,0.382830,0,4,0,4
1,Vanilla Mistral 7B zero-shot,P6,180,0.470353,0,4,0,4
2,Vanilla Mistral 7B zero-shot,P7,157,0.632249,6,24,0,28
3,Vanilla Mistral 7B zero-shot,P8,72,0.041013,24,50,15,58


Macro Avg QWK P5-P8: 0.38161139361085805
Saved: /content/drive/MyDrive/ielts/IELTS-Writing-Evals/vanilla_baseline_outputs_p5_p8/vanilla_mistral_p5_p8_qwk_metrics.csv


In [11]:

# Dòng này để partner copy vào bảng
table_row = {f"P{s}": metrics_df.loc[metrics_df["essay_set"] == f"P{s}", "qwk"].iloc[0] for s in ESSAY_SETS_TO_RUN}
table_row["Avg_P5_P8"] = avg_qwk
pd.DataFrame([table_row])


,P5,P6,P7,P8,Avg_P5_P8
0,0.38283,0.470353,0.632249,0.041013,0.381611



## 8. Debug nếu kết quả lạ


In [12]:

debug_cols = ["essay_set", "gold", "pred", "raw_generation", "essay"]
pd.set_option("display.max_colwidth", 300)
pred_df[debug_cols].sample(min(10, len(pred_df)), random_state=RANDOM_SEED)


,essay_set,gold,pred,raw_generation,essay
414,7,8.0,10,<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nWrite about patience. Being patient means that you are understanding and tolerant. A patient person experiences difficulties without complaining.\n\nDo only one of the following: write a story about a time when you were patient OR write...,One time I was pacient was when my mum said that she would buy me a @ORGANIZATION1 @CAPS1 I waited and waited and waited and wated for her but I knew I had to be pacient because she was going through hard times.
525,5,3.0,4,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nRead the memoir ""Narciso Rodriguez"" from Home: The Blueprints of Our Lives.\n\nDescribe the mood created by the author in the memoir. Support your answer with relevant and specific information from the memoir.\n\nRubric summary:\nScore ...","In the memoir, ""Narciso Rodriguez"" from Home: The Blueprints of Our Lives. Narciso projects a very greatful, thankful + optimistic mood. Every sentence, he thanks a specific person for bringing such greatness to his life. In paragraph six, Narciso says, ""I will always be greatful to my parents f..."
43,7,20.0,25,<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nWrite about patience. Being patient means that you are understanding and tolerant. A patient person experiences difficulties without complaining.\n\nDo only one of the following: write a story about a time when you were patient OR write...,"Two summers ago, I helped out with kids camp at my church. I helped with the preschoolers for the week and that required a lot of patience. They were very exited and roudy. They were always running around and shouting. I had to keep a calm voice and be patient with the children or it would make ..."
193,7,22.0,22,<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nWrite about patience. Being patient means that you are understanding and tolerant. A patient person experiences difficulties without complaining.\n\nDo only one of the following: write a story about a time when you were patient OR write...,"I was at the detrait airport. It was around @NUM1 am,the sky was bright and gloomy but no rain, just a warm breeze blowing in my face. My mom,brother and @CAPS1 through the tall metal doors and saw everything from gift shops to our destiny where we would @NUM2.We got our tickets and headed off i..."
186,8,28.0,52,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nWe all understand the benefits of laughter. For example, someone once said, ""Laughter is the shortest distance between two people."" Many other people believe that laughter is an important part of any relationship.\n\nTell a true story i...","@PERSON1 The @CAPS1 @CAPS2 @CAPS3 out with buddies are some of the best @CAPS2 of my life if its an adventure or just a @CAPS1 time @CAPS3 out. I wen stergion fishing for the first time in my life with my buddy @PERSON2, i haven't known him vary long because i just moved to a new school so i jus..."
34,5,4.0,4,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nRead the memoir ""Narciso Rodriguez"" from Home: The Blueprints of Our Lives.\n\nDescribe the mood created by the author in the memoir. Support your answer with relevant and specific information from the memoir.\n\nRubric summary:\nScore ...","In Narciso Rodriguez's memoir, Narciso Rodriguez from Home: the Blueprints of Our Lives, the mood created by the author is appreciative. Narciso Rodriguez is deeply thankful of his parents and other friends and family for helping him through his life.Narciso Rodriguez repeats several times throu..."
128,6,2.0,4,"<s>[INST] You are an expert essay scorer.\n\nEssay prompt:\nRead the excerpt ""The Mooring Mast"" by Marcia Amidon Lusted.\n\nBased on the excerpt, describe the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. Support your answer with relev...","The architects building the